# Organic Reaction-Mechanism Specialist — Train & Benchmark (Colab GPU)

Fine-tune a small **Qwen** model on the *decontaminated* mechanism dataset built by the
`rxndata` pipeline, then score it on the held-out **oMe-Gold** test set with the **exact
oMeS metric** the oMeBench paper uses — so the number is directly comparable to frontier
models (Gemini, GPT-5.x, Claude, …).

**What this notebook does**
1. Build the training set **offline** from the committed Tier-A files (oMe-Silver + oMe-Template),
   run the validate → **decontaminate** → dedup → format phases.
2. SFT a Qwen base model with **LoRA** on `mechanism_full` targets.
3. Evaluate on **oMe-Gold (196 rxns, never seen in training)** with the real oMeS scorer.
4. Drop the score into a **leaderboard vs frontier models**.

> **Runtime → change type to GPU.** `Runtime ▸ Change runtime type ▸ T4 GPU` (free) or A100/L4 (Pro).

### ⏱️ Per-cell time estimates

| # | Cell | T4 (free) | A100 / L4 (Pro) |
|---|------|-----------|-----------------|
| 1 | GPU / runtime check | <5 s | <5 s |
| 2 | Install dependencies | 3–6 min | 3–6 min |
| 3 | Get the repo | 10–30 s | 10–30 s |
| 4 | Build decontaminated dataset | 3–5 min | 3–5 min |
| 5 | Inspect dataset + token lengths | 20–40 s | 20–40 s |
| 6 | Config knobs | <5 s | <5 s |
| 7 | Load base model + tokenizer | 1–3 min (download) | 1–3 min |
| 8 | **Train (LoRA SFT)** | **QUICK ~20 min / full ~60–110 min** | **QUICK ~6 min / full ~12–25 min** |
| 9 | Evaluate fine-tuned on oMe-Gold | QUICK ~8 min / full ~25–40 min | QUICK ~3 min / full ~6–12 min |
| 10 | (optional) Evaluate the base model | same as #9 | same as #9 |
| 11 | Leaderboard vs frontier | <5 s | <5 s |
| 12 | (optional) Live frontier eval via API | ~5–15 min | ~5–15 min |

`QUICK_MODE = True` (cell 6) subsets training/eval for a fast end-to-end pass; set it `False`
for the full, reportable run.


## 1 · GPU / runtime check  ·  ⏱️ <5 s
If this prints `No GPU`, enable one via *Runtime ▸ Change runtime type*.

In [ ]:
import subprocess, torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap  = torch.cuda.get_device_capability(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} | compute {cap[0]}.{cap[1]} | VRAM {vram:.0f} GB")
    # bf16 needs Ampere+ (compute >= 8.0); T4 is 7.5 -> use fp16.
    BF16 = cap[0] >= 8
    print("Using", "bf16" if BF16 else "fp16")
else:
    BF16 = False
    print("No GPU — set Runtime ▸ Change runtime type ▸ GPU before continuing.")

## 2 · Install dependencies  ·  ⏱️ 3–6 min
Colab already ships `torch`. We add RDKit (validity/oMeS), transformers/peft/accelerate
(training), and datasets. Pinned for reproducibility.

In [ ]:
%pip -q install \
  "rdkit==2025.9.2" \
  "transformers==4.55.2" \
  "peft==0.13.2" \
  "accelerate==1.10.1" \
  "datasets==3.6.0" \
  "pyyaml==6.0.2" "polars==1.36.1" "pyarrow==17.0.0" "tqdm==4.67.1"
print("deps installed — if Colab asks to RESTART the runtime, do it, then re-run from cell 3.")
import transformers, accelerate, peft
print("transformers", transformers.__version__, "| accelerate", accelerate.__version__, "| peft", peft.__version__)

## 3 · Get the repo  ·  ⏱️ 10–30 s
Point `REPO_URL` at your clone of this project (it contains the pipeline + the oMeBench data
+ the oMeS scorer). If the repo is private, either make a public mirror, use a token URL, or
upload it and set `REPO_DIR` to the uploaded path.

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/YOURNAME/SLM-1.git"   # <-- EDIT ME
REPO_DIR = "/content/SLM-1"

if not os.path.isdir(REPO_DIR):
    try:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    except Exception as e:
        print("git clone failed:", e)
        print("Alternative: upload the repo folder to /content/SLM-1, or mount Drive:")
        print("  from google.colab import drive; drive.mount('/content/drive')")
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
sys.path.insert(0, REPO_DIR)
os.environ["PYTHONPATH"] = os.path.join(REPO_DIR, "src")
print("repo:", REPO_DIR)
print("data files present:", sorted(p.name for p in pathlib.Path("data").glob("oMe_*")))

## 4 · Build the decontaminated training set  ·  ⏱️ 3–5 min
Runs the pipeline **offline** using only the committed Tier-A files (oMe-Silver + oMe-Template):

`ingest → normalize → decompose/type → validate → decontaminate(vs oMe-Gold) → dedup → format`

We skip Tier-B/C ingest (needs network) and atom-mapping (RXNMapper, not needed for the SFT
mechanism targets). The key guarantee — **zero oMe-Gold overlap** — still runs (phase 6).

In [ ]:
import subprocess, os, json
env = dict(os.environ, PYTHONPATH=os.path.join(REPO_DIR, "src"))
def run_phase(mod, *args):
    print(f"\n=== {mod} {' '.join(args)} ===")
    r = subprocess.run([sys.executable, "-m", mod, *args],
                       env=env, capture_output=True, text=True)
    # show the gate summary lines
    for line in (r.stdout + r.stderr).splitlines():
        if any(k in line for k in ("GATE","records","PASS","FAIL","removed","kept",
                                   "leaks","train /","self-check","tokenizer","Tier")):
            print(line)
    if r.returncode != 0:
        print(r.stderr[-2000:]); raise RuntimeError(f"{mod} failed")

run_phase("rxndata.ontology")
run_phase("rxndata.phase1_ingest", "--only", "ome_silver", "ome_template")
run_phase("rxndata.phase2_normalize", "--only", "ome_silver", "ome_template")
run_phase("rxndata.phase3_mechanism")
run_phase("rxndata.phase5_validate")
run_phase("rxndata.phase6_decontaminate")     # <-- proves 0 gold leaks
run_phase("rxndata.phase7_dedup")
run_phase("rxndata.phase8_format", "--style", "cot")
run_phase("rxndata.phase9_splits")
print("\nDONE — dataset in data/final/")

## 5 · Inspect dataset + build train/val splits  ·  ⏱️ 20–40 s
We split the SFT rows using the pipeline's *stratified, decontaminated* id manifests, and
sanity-check token lengths against the Qwen tokenizer.

In [ ]:
import json, pathlib
from collections import Counter

sft_path = "data/final/sft_mechanisms.jsonl"
rows = [json.loads(l) for l in open(sft_path)]
mech_full = [r for r in rows if r["task"] == "mechanism_full"]
print(f"SFT rows: {len(rows)} | mechanism_full: {len(mech_full)}")
print("tasks:", dict(Counter(r["task"] for r in rows)))

# Decontamination proof
decon = json.load(open("data/final/decontamination_report.json"))
print("\nDECONTAMINATION vs oMe-Gold:",
      "removed", decon["removed"],
      "| gold InChIKey leaks:", decon.get("post_check_gold_inchikey_leaks"))

# Stratified train/val split from the pipeline manifests (falls back to random).
try:
    train_ids = set(json.load(open("data/final/splits/train_ids.json")))
    val_ids   = set(json.load(open("data/final/splits/val_ids.json")))
    train_rows = [r for r in mech_full if r["meta"]["reaction_id"] in train_ids]
    val_rows   = [r for r in mech_full if r["meta"]["reaction_id"] in val_ids]
    if not train_rows:
        raise ValueError("empty")
except Exception:
    import random; random.seed(0); rr = mech_full[:]; random.shuffle(rr)
    n_val = max(1, len(rr)//20); val_rows, train_rows = rr[:n_val], rr[n_val:]
print(f"train: {len(train_rows)} | val: {len(val_rows)}")

# One rendered example
ex = train_rows[0]
print("\n--- example (user, truncated) ---\n", ex["messages"][1]["content"][:300])
print("\n--- assistant target (tail) ---\n", ex["messages"][2]["content"][-200:])

## 6 · Config  ·  ⏱️ <5 s
`QUICK_MODE=True` gives a fast full pass (subset + 1 epoch). Set `False` for the reportable run.
On a **T4**, if you hit OOM use `Qwen/Qwen2.5-0.5B-Instruct` or lower `MAX_SEQ_LEN`.

In [ ]:
QUICK_MODE      = True                         # False = full, reportable run

BASE_MODEL      = "Qwen/Qwen2.5-1.5B-Instruct" # T4-safe with LoRA; use 0.5B if OOM
MAX_SEQ_LEN     = 3072                          # p95 of our data is ~2564 tokens
EPOCHS          = 1 if QUICK_MODE else 3
PER_DEVICE_BATCH= 2
GRAD_ACCUM      = 8                             # effective batch = 16
LR              = 2e-4                           # LoRA lr
LORA_R          = 16
LORA_ALPHA      = 32
MAX_TRAIN       = 600 if QUICK_MODE else None    # cap training rows in quick mode
EVAL_LIMIT      = 60  if QUICK_MODE else 196      # oMe-Gold reactions to score
EVAL_MAX_NEW    = 1024                            # generation budget per reaction
OUT_DIR         = "/content/ckpt-mech-lora"
print(dict(QUICK_MODE=QUICK_MODE, BASE_MODEL=BASE_MODEL, EPOCHS=EPOCHS,
           MAX_TRAIN=MAX_TRAIN, EVAL_LIMIT=EVAL_LIMIT))

## 7 · Load base model + tokenizer  ·  ⏱️ 1–3 min (first download)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if BF16 else torch.float16,
    device_map={"": 0},
    trust_remote_code=True,
)
model.config.use_cache = False
print("loaded", BASE_MODEL, "| params:", sum(p.numel() for p in model.parameters())/1e9, "B")

## 8 · Train — LoRA SFT  ·  ⏱️ T4: QUICK ~20 min / full ~60–110 min · A100/L4: QUICK ~6 / full ~12–25 min

Robust to TRL version drift: plain `transformers.Trainer` with **prompt-masked labels**
(loss only on the assistant mechanism tokens). Watch the loss fall — a healthy run drops from
~1.5 to ~0.2–0.4.

In [ ]:
import torch
from dataclasses import dataclass
from typing import List, Dict
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel

# ---- prompt-masked tokenization (loss on assistant span only) ----
def encode(example):
    msgs = example["messages"]
    prompt_ids = tok.apply_chat_template(msgs[:-1], add_generation_prompt=True, tokenize=True)
    full_ids   = tok.apply_chat_template(msgs,      add_generation_prompt=False, tokenize=True)
    labels = [-100]*len(prompt_ids) + full_ids[len(prompt_ids):]
    full_ids, labels = full_ids[:MAX_SEQ_LEN], labels[:MAX_SEQ_LEN]
    return {"input_ids": full_ids, "labels": labels}

class ChatDS(Dataset):
    def __init__(self, rows): self.rows = [encode(r) for r in rows]
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

@dataclass
class Collator:
    pad_id: int
    def __call__(self, feats: List[Dict]):
        m = max(len(f["input_ids"]) for f in feats)
        import torch
        ids, lab, att = [], [], []
        for f in feats:
            n = m - len(f["input_ids"])
            ids.append(f["input_ids"] + [self.pad_id]*n)
            lab.append(f["labels"]    + [-100]*n)
            att.append([1]*len(f["input_ids"]) + [0]*n)
        return {"input_ids": torch.tensor(ids), "labels": torch.tensor(lab),
                "attention_mask": torch.tensor(att)}

_train = train_rows[:MAX_TRAIN] if MAX_TRAIN else train_rows
train_ds, val_ds = ChatDS(_train), ChatDS(val_rows)
print(f"tokenized train={len(train_ds)} val={len(val_ds)}")

# ---- LoRA ----
# Idempotent + portable: skip if an adapter is already attached (safe re-run),
# and name Qwen's projection layers explicitly instead of the "all-linear"
# shorthand (which requires a raw PreTrainedModel and breaks on re-wrap).
if isinstance(model, PeftModel):
    print("A LoRA adapter is already attached to `model` — reusing it. "
          "To start fresh, re-run the 'Load base model' cell first.")
else:
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

args = TrainingArguments(
    output_dir=OUT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, warmup_ratio=0.03, lr_scheduler_type="cosine",
    logging_steps=10, save_strategy="no",
    bf16=BF16, fp16=not BF16, gradient_checkpointing=True,
    report_to="none", optim="adamw_torch",
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  data_collator=Collator(tok.pad_token_id))
trainer.train()
model.save_pretrained(OUT_DIR); tok.save_pretrained(OUT_DIR)
print("saved LoRA adapter ->", OUT_DIR)

## 9 · Evaluate the fine-tuned model on oMe-Gold  ·  ⏱️ T4 QUICK ~8 min / full ~25–40 min

Uses the repo's **exact oMeS scorer** (`omebench_eval.scoring.oMeS`) on the held-out gold set,
with the same `default.txt` prompt the benchmark uses. Batched generation for speed; the metric
is identical to the paper's, so the S_partial here is comparable to the frontier table below.

In [ ]:
import json, torch, time
from omebench_eval.scoring import oMeS, canonical_smiles
from omebench_eval.parsing import extract_mechanism
from omebench_eval.dataset import load_dataset, load_prompt_template, build_prompt

SYSTEM = "You are an expert in organic reaction mechanisms."
gold = load_dataset("gold")[:EVAL_LIMIT]
template = load_prompt_template("default")

def score_model(m, limit_rows, batch=8, max_new=EVAL_MAX_NEW, label="model"):
    m.eval(); tok.padding_side = "left"
    prompts, ids = [], []
    for r in limit_rows:
        u = build_prompt(template, r["reactants_smiles"], r["products_smiles"], r.get("conditions"))
        text = tok.apply_chat_template(
            [{"role":"system","content":SYSTEM},{"role":"user","content":u}],
            tokenize=False, add_generation_prompt=True)
        prompts.append(text); ids.append(r["reaction_id"])
    outs = {}
    t0 = time.time()
    for i in range(0, len(prompts), batch):
        chunk = prompts[i:i+batch]
        enc = tok(chunk, return_tensors="pt", padding=True, truncation=True,
                  max_length=MAX_SEQ_LEN).to(m.device)
        with torch.no_grad():
            gen = m.generate(**enc, max_new_tokens=max_new, do_sample=False,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
        for j, seq in enumerate(gen):
            new = seq[enc.input_ids.shape[1]:]
            outs[ids[i+j]] = tok.decode(new, skip_special_tokens=True)
        print(f"  [{label}] {min(i+batch,len(prompts))}/{len(prompts)}  "
              f"({time.time()-t0:.0f}s)", end="\r")
    print()
    # score with oMeS
    per = []
    for r in limit_rows:
        mech = extract_mechanism(outs.get(r["reaction_id"]))
        if not isinstance(mech, list):
            per.append({"level": r.get("level"), "S_total":0.0,"S_partial":0.0,"V":0,"L":0}); continue
        pred = [(s.get("subtype"), s.get("intermediate_smiles","")) for s in mech if isinstance(s,dict)]
        g = [(s["subtype"], s["intermediate_smiles"], s["step_weight"]) for s in r["mechanism"]]
        try: res = oMeS(g, pred); per.append({"level":r.get("level"),"S_total":res.S_total,
                                              "S_partial":res.S_partial,"V":res.V,"L":res.L})
        except Exception: per.append({"level":r.get("level"),"S_total":0.0,"S_partial":0.0,"V":0,"L":0})
    def avg(k, rows=per): return round(sum(x[k] for x in rows)/max(1,len(rows)),4)
    summary = {"model":label,"n":len(per),"S_partial":avg("S_partial"),"S_total":avg("S_total"),
               "V":avg("V"),"L":avg("L")}
    for lvl in ("easy","medium","hard"):
        sub=[x for x in per if x["level"]==lvl]
        if sub: summary[f"S_partial_{lvl}"]=round(sum(x['S_partial'] for x in sub)/len(sub),4)
    return summary

ft = score_model(model, gold, label="qwen-mech-lora")
print("\nFINE-TUNED:", json.dumps(ft, indent=2))

## 10 · (optional) Evaluate the **base** model for a delta  ·  ⏱️ same as #9
Shows how much the fine-tune moved the needle. Reloads the base weights (without the adapter).
Skip to save time if you only need the leaderboard slot.

In [ ]:
RUN_BASE_EVAL = True   # set False to skip

base_summary = None
if RUN_BASE_EVAL:
    from transformers import AutoModelForCausalLM
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.bfloat16 if BF16 else torch.float16,
        device_map={"":0}, trust_remote_code=True)
    base_summary = score_model(base, gold, label=f"{BASE_MODEL.split('/')[-1]} (base)")
    print("\nBASE:", json.dumps(base_summary, indent=2))
    del base; torch.cuda.empty_cache()

## 11 · Leaderboard vs frontier models  ·  ⏱️ <5 s

Frontier oMeS **S_partial on oMe-Gold** (0–1 scale). Sources: oMeBench paper (arXiv:2510.07731)
and this repo's own API-harness runs (see `README.md` / `BRAINLIFT.md`). Our fine-tuned row is
inserted from cell 9.

> Reality check (from the paper's feasibility notes): a **1.5 B** specialist won't beat GPT-5.x;
> the honest target is to **clear the weak/mid baselines** (GPT-4o 0.05, Sonnet-4 0.18) and approach
> the paper's 4 B specialist (~0.20–0.30) on in-domain mechanisms — while *proving no test leakage*.

In [ ]:
FRONTIER = [
    ("Gemini-3.1-Pro",        0.51,  "paper/README"),
    ("GPT-5.5 (this harness)",0.469, "repo run"),
    ("Gemini-Pro-2.5",        0.379, "paper"),
    ("GPT-5",                 0.291, "paper"),
    ("o3",                    0.281, "paper"),
    ("paper 4B SFT specialist",0.30, "paper (ICL)"),
    ("DeepSeek-R1 (685B)",    0.25,  "paper"),
    ("Claude-Sonnet-4",       0.179, "paper"),
    ("GPT-4o",                0.05,  "paper"),
    ("Qwen-3-4B (untuned)",   0.042, "paper"),
    ("LLaMA-3-8B (untuned)",  0.033, "paper"),
]
board = list(FRONTIER)
board.append((ft["model"] + "  ⭐ (ours)", ft["S_partial"], f"this notebook, n={ft['n']}"))
if base_summary:
    board.append((base_summary["model"], base_summary["S_partial"], f"this notebook, n={base_summary['n']}"))
board.sort(key=lambda x: -x[1])

print(f"{'model':<34}{'S_partial':>10}   source")
print("-"*70)
for name, sp, src in board:
    star = "  <<<" if "ours" in name else ""
    print(f"{name:<34}{sp:>10.3f}   {src}{star}")

print("\nOur model, by difficulty:")
for lvl in ("easy","medium","hard"):
    k=f"S_partial_{lvl}"
    if k in ft: print(f"  {lvl:<7} {ft[k]:.3f}")
print(f"\nValidity(V)={ft['V']}  LogicalFidelity(L)={ft['L']}  S_total={ft['S_total']}")

## 12 · (optional) Live frontier eval via API  ·  ⏱️ ~5–15 min
Reproduce a frontier number yourself instead of trusting the table. Needs an API key and spend.
Uses the repo's API harness (identical oMeS scorer).

In [ ]:
RUN_API_EVAL = False   # set True and add a key

if RUN_API_EVAL:
    import os, subprocess
    os.environ["OPENAI_API_KEY"] = ""     # <-- your key
    # or: os.environ["ANTHROPIC_API_KEY"] = "..."
    %pip -q install openai anthropic
    # score 40 gold reactions with gpt-5.5 (edit --models / --limit as you like)
    !python -m omebench_eval.cli run --models gpt-5.5 --dataset gold --prompt cot --limit 40 --max-tokens 16000
    !python -m omebench_eval.cli report --dataset gold

## Next steps — push past SFT with RL

The dataset carries a `reference` field ([subtype, canonical_smiles, weight]) on every row, so
the repo's **GRPO** loop (`training/grpo_train.py` + `training/reward.py`) can optimize directly
against the verifiable oMeS reward — the lever the oMeBench paper left untapped. After this SFT
checkpoint, continue with GRPO for the biggest expected gain.

**Honesty notes**
- We evaluate on **oMe-Gold**, held out and *proven* decontaminated (cell 4/5). No leakage.
- oMeBench measures *mechanism* reasoning specifically; it is the organic-chemistry benchmark this
  dataset targets. Broad chemistry knowledge (ChemBench, GPQA) is out of scope for a 1.5 B mechanism
  specialist.
- In-domain scores (same named reactions, new substituents) run higher than truly novel mechanisms —
  interpret a leaderboard win as *data/param-efficiency*, not general chemical mastery.
